#  ParkLens: Classical Computer Vision Parking Occupancy Detector
### Course: *Fundamentals of Computer Vision* | University Mini-Project

This notebook provides a step-by-step interactive walkthrough of the **ParkLens** classical computer vision pipeline.

**Core Philosophy:** Explainable, mathematically transparent classical computer vision (**No deep learning, No YOLO, No PyTorch/TF**).

---
### Pipeline Stages:
1. **Image Acquisition & Ingestion:** Loading PKLot images and polygon slot annotations.
2. **Perspective Homography Rectification:** Warping irregular quadrilaterals to $64 \times 64$ patches.
3. **Noise Filtering & Illumination Normalization:** Gaussian blur + CLAHE (Contrast-Limited Adaptive Histogram Equalization).
4. **Feature Extraction:**
   - Canny Edge Density
   - Background Subtraction Difference (vs. empty canonical slot patch)
   - Laplacian Variance (High-frequency content / texture dispersion)
   - Luminance Standard Deviation
5. **Decision Logic & Confidence Formulation:** Weighted linear combination vs. empirical decision threshold.
6. **Full-Lot Visualization:** Overlaying predictions, metrics, and error highlighting.

In [ ]:
# Environment Setup
import sys
from pathlib import Path

# Ensure project root is in path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import json
import pickle

from config import DATA_DIR, INDEX_FILE, PARAMS_FILE, OUTPUTS_DIR
from src.acquisition import load_dataset_index, parse_pklot_xml
from src.preprocessing import preprocess_patch, to_grayscale, apply_blur, apply_clahe
from src.features import extract_features, edge_density, background_diff_score
from src.classifier import rule_based_classify, classify_spaces_in_image, load_tuned_params
from src.visualization import draw_parking_overlay, make_pipeline_figure
from src.utils import warp_polygon_to_patch

print(" ParkLens modules imported successfully!")

## 1. Load Dataset Index & Select Sample Image

In [ ]:
# Load indexed dataset
index = load_dataset_index(INDEX_FILE)
print(f"Total indexed images: {len(index)}")

# Select a sample image from PUCPR lot under Sunny conditions
sample_record = next(r for r in index if r["lot"] == "PUCPR" and r["weather"] == "Sunny")
img_path = PROJECT_ROOT / sample_record["image_path"]
img_bgr = cv2.imread(str(img_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

print(f"Loaded Image: {sample_record['image_path']}")
print(f"Resolution: {img_bgr.shape[1]}x{img_bgr.shape[0]} | Parking Spaces: {len(sample_record['spaces'])}")

plt.figure(figsize=(10, 6))
plt.imshow(img_rgb)
plt.title(f"Raw PKLot Image – {sample_record['lot']} ({sample_record['weather']})")
plt.axis("off")
plt.show()

## 2. Perspective Homography Rectification
Quadrilateral parking slots are skewed due to camera angle perspective. We use `cv2.getPerspectiveTransform` and `cv2.warpPerspective` to project them onto a canonical $64 \times 64$ patch.

In [ ]:
# Pick one occupied space and one free space
free_sp = next(s for s in sample_record["spaces"] if s["occupied"] == 0)
occ_sp = next(s for s in sample_record["spaces"] if s["occupied"] == 1)

patch_free = warp_polygon_to_patch(img_bgr, free_sp["polygon"])
patch_occ = warp_polygon_to_patch(img_bgr, occ_sp["polygon"])

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(cv2.cvtColor(patch_free, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Free Space #{free_sp['id']}")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(patch_occ, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Occupied Space #{occ_sp['id']}")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 3. Preprocessing: Gaussian Blur + CLAHE
CLAHE (*Contrast-Limited Adaptive Histogram Equalization*) prevents over-amplification of noise while equalizing local regions against intense sunlight or shadows.

In [ ]:
prep_occ = preprocess_patch(patch_occ, blur_k=5, clahe_clip=2.0)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(cv2.cvtColor(prep_occ["bgr"], cv2.COLOR_BGR2RGB))
axes[0].set_title("Original Patch")

axes[1].imshow(prep_occ["gray"], cmap="gray")
axes[1].set_title("Grayscale")

axes[2].imshow(prep_occ["blurred"], cmap="gray")
axes[2].set_title("Gaussian Blur (5x5)")

axes[3].imshow(prep_occ["clahe"], cmap="gray")
axes[3].set_title("CLAHE Enhanced")

for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Feature Extraction: Canny Edge Detection & Background Subtraction

In [ ]:
# Load precomputed empty reference patches
ref_path = DATA_DIR / "ref_patches.pkl"
with open(ref_path, "rb") as f:
    ref_patches = pickle.load(f)

ref_patch = ref_patches.get(f"{sample_record['lot']}_{occ_sp['id']}")

# Extract features
feat_free = extract_features(patch_free, ref_patches.get(f"{sample_record['lot']}_{free_sp['id']}"))
feat_occ = extract_features(patch_occ, ref_patch)

print("=== FEATURE COMPARISON ===")
print(f"Feature                 Free Space #{free_sp['id']:<4} Occupied Space #{occ_sp['id']}")
print("-" * 55)
print(f"Edge Density:           {feat_free['edge_density']:.4f}         {feat_occ['edge_density']:.4f}")
print(f"Diff Score:             {feat_free['diff_score']:.4f}         {feat_occ['diff_score']:.4f}")
print(f"Laplacian Variance:     {feat_free['laplacian_variance']:.1f}           {feat_occ['laplacian_variance']:.1f}")
print(f"Luminance Std Dev:      {feat_free['gray_std']:.2f}            {feat_occ['gray_std']:.2f}")

# Visualize edge map and difference map
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(feat_occ["edge_map"], cmap="gray")
axes[0].set_title(f"Canny Edges (Occupied #{occ_sp['id']})")
axes[0].axis("off")

axes[1].imshow(feat_occ["diff_map"], cmap="hot")
axes[1].set_title(f"Background Diff Map (Occupied #{occ_sp['id']})")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 5. End-to-End Image Classification & Annotation

In [ ]:
# Load tuned parameters for PUCPR Sunny
params = load_tuned_params(PARAMS_FILE, lot="PUCPR", weather="Sunny")
print("Tuned Parameters Loaded:", params)

# Classify all parking spaces in the image
results = classify_spaces_in_image(img_bgr, sample_record["spaces"], "PUCPR", ref_patches, params)

total = len(results)
occupied = sum(1 for r in results if r["prediction"] == 1)
free = total - occupied

# Evaluate current frame accuracy
correct = sum(1 for r in results if r["prediction"] == r["ground_truth"])
acc = correct / total * 100

print(f"\nDetection Results:")
print(f"  Total Spaces:    {total}")
print(f"  Free Spots:      {free}")
print(f"  Occupied Spots:  {occupied}")
print(f"  Frame Accuracy:  {acc:.1f}% ({correct}/{total} correct)")

# Render full-lot overlay
annotated_bgr = draw_parking_overlay(img_bgr, results, show_ids=False, highlight_errors=True)
annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 8))
plt.imshow(annotated_rgb)
plt.title(f"ParkLens Output – Free: {free} | Occupied: {occupied} | Accuracy: {acc:.1f}%")
plt.axis("off")
plt.show()

## 6. Test Set Benchmark Plots
Loading quantitative evaluation results produced by `python main.py evaluate` on the unseen 20% test split.

In [ ]:
with open(OUTPUTS_DIR / "metrics.json", "r") as f:
    metrics = json.load(f)

print("Overall Test Split Performance Metrics:")
for k, v in metrics["overall"].items():
    if isinstance(v, float):
        print(f"  {k:20s}: {v*100:.2f}%")
    else:
        print(f"  {k:20s}: {v}")

# Display saved artifact plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cm_img = plt.imread(str(OUTPUTS_DIR / "confusion_matrix.png"))
roc_img = plt.imread(str(OUTPUTS_DIR / "roc_curves.png"))
w_img = plt.imread(str(OUTPUTS_DIR / "weather_accuracy.png"))

axes[0].imshow(cm_img); axes[0].set_title("Confusion Matrix Heatmap"); axes[0].axis("off")
axes[1].imshow(roc_img); axes[1].set_title("ROC Curves & AUC"); axes[1].axis("off")
axes[2].imshow(w_img); axes[2].set_title("Weather Accuracy Comparison"); axes[2].axis("off")

plt.tight_layout()
plt.show()